In [57]:
import json
import google.generativeai as genai
from typing import Dict, List, Tuple
import numpy as np
from difflib import SequenceMatcher
import pandas as pd
import time

In [ ]:
# Configure Gemini
genai.configure(api_key='')
model = genai.GenerativeModel('gemini-2.0-flash')

In [59]:

EXTRACTION_PROMPT = """Analyze this story and extract symbolic narrative elements in JSON format.

Story: {story}

Extract the following elements:

1. **Abstract Theme**: The core ideas, motifs, moral lessons, or philosophical concepts (2-4 keywords)
2. **Key Events**: The main sequence of events/actions in chronological order (4-8 events as short phrases)
3. **Outcomes**: The final results or resolutions (2-3 outcome types)
4. **Character Roles**: Main character archetypes (e.g., protagonist-hero, antagonist-villain, mentor, victim)
5. **Conflict Type**: The nature of the central conflict (e.g., person-vs-nature, person-vs-society, internal-conflict)

Return ONLY valid JSON in this exact format:
{{
    "themes": ["theme1", "theme2", "theme3"],
    "events": ["event1", "event2", "event3", "event4"],
    "outcomes": ["outcome1", "outcome2"],
    "character_roles": ["role1", "role2"],
    "conflict_type": "conflict_type"
}}

Be concise and focus on narrative structure, not surface details."""

In [ ]:


def extract_symbolic_elements(story, max_retries=3):
    """Extract symbolic narrative elements using Gemini."""
    for attempt in range(max_retries):
        try:
            prompt = EXTRACTION_PROMPT.format(story=story)
            response = model.generate_content(prompt)
            
            # Clean response
            text = response.text.strip()
            # Remove markdown code blocks if present
            if text.startswith("```json"):
                text = text[7:]
            if text.startswith("```"):
                text = text[3:]
            if text.endswith("```"):
                text = text[:-3]
            text = text.strip()
            
            elements = json.loads(text)
            
            # Validate structure
            required_keys = ["themes", "events", "outcomes", "character_roles", "conflict_type"]
            if all(key in elements for key in required_keys):
                return elements
            else:
                print(f"Missing keys in response, attempt {attempt + 1}")
                
        except json.JSONDecodeError as e:
            print(f"JSON decode error on attempt {attempt + 1}: {e}")
            time.sleep(1)
        except Exception as e:
            print(f"Error on attempt {attempt + 1}: {e}")
            time.sleep(1)
    
    # Return empty structure if all retries fail
    return {
        "themes": [],
        "events": [],
        "outcomes": [],
        "character_roles": [],
        "conflict_type": ""
    }


In [ ]:

def list_similarity(list1, list2):
    """Calculate similarity between two lists of strings using token overlap."""
    if not list1 or not list2:
        return 0.0
    
    # Normalize to lowercase
    set1 = set([item.lower() for item in list1])
    set2 = set([item.lower() for item in list2])
    
    # Jaccard similarity
    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))
    
    if union == 0:
        return 0.0
    
    return intersection / union

In [ ]:
def sequence_similarity(seq1, seq2):
    """Calculate similarity between two sequences considering order."""
    if not seq1 or not seq2:
        return 0.0
    
    # Use SequenceMatcher for ordered similarity
    matcher = SequenceMatcher(None, 
                             [e.lower() for e in seq1], 
                             [e.lower() for e in seq2])
    return matcher.ratio()


In [ ]:
def string_similarity(str1, str2):
    """Calculate similarity between two strings."""
    if not str1 or not str2:
        return 0.0
    
    matcher = SequenceMatcher(None, str1.lower(), str2.lower())
    return matcher.ratio()

In [ ]:
def compute_symbolic_similarity(elements1, elements2):
    """
    Compute overall symbolic similarity based on three narrative components:
    - Abstract Theme (30% weight)
    - Course of Action/Events (40% weight) 
    - Outcomes (30% weight)
    """
    
    # Theme similarity (using Jaccard on themes)
    theme_sim = list_similarity(elements1.get("themes", []), 
                                elements2.get("themes", []))
    
    # Events similarity (using sequence matching - order matters)
    events_sim = sequence_similarity(elements1.get("events", []), 
                                    elements2.get("events", []))
    
    # Outcomes similarity (using Jaccard)
    outcomes_sim = list_similarity(elements1.get("outcomes", []), 
                                   elements2.get("outcomes", []))
    
    # Additional features (lower weight)
    roles_sim = list_similarity(elements1.get("character_roles", []), 
                               elements2.get("character_roles", []))
    
    conflict_sim = string_similarity(elements1.get("conflict_type", ""), 
                                    elements2.get("conflict_type", ""))
    
    # Weighted combination matching task definition
    total_similarity = (
        0.25 * theme_sim +          # Abstract Theme
        0.35 * events_sim +          # Course of Action (most important)
        0.25 * outcomes_sim +        # Outcomes
        0.10 * roles_sim +           # Character roles (supplementary)
        0.05 * conflict_sim          # Conflict type (supplementary)
    )
    
    return total_similarity

In [ ]:
def process_triplet(anchor, text_a, text_b):
    """
    Process a triplet and determine which text is closer to anchor.
    Returns (text_a_is_closer, debug_info)
    """
    print("Extracting symbolic elements...")
    
    # Extract elements
    anchor_elements = extract_symbolic_elements(anchor)
    a_elements = extract_symbolic_elements(text_a)
    b_elements = extract_symbolic_elements(text_b)
    
    # Compute similarities
    sim_a = compute_symbolic_similarity(anchor_elements, a_elements)
    sim_b = compute_symbolic_similarity(anchor_elements, b_elements)
    
    text_a_is_closer = sim_a > sim_b
    
    debug_info = {
        "anchor_elements": anchor_elements,
        "a_elements": a_elements,
        "b_elements": b_elements,
        "similarity_a": sim_a,
        "similarity_b": sim_b,
        "text_a_is_closer": text_a_is_closer
    }
    
    return text_a_is_closer, debug_info

In [ ]:
def evaluate_validation(data):
    """Evaluate on validation set with ground truth labels."""
    correct = 0
    total = len(data)
    
    for idx, item in enumerate(data):
        print(f"\nValidation {idx + 1}/{total}...")
        
        try:
            text_a_is_closer, debug_info = process_triplet(
                item["anchor_text"],
                item["text_a"],
                item["text_b"]
            )
            
            if text_a_is_closer == item["text_a_is_closer"]:
                correct += 1
            
            print(f"Prediction: {text_a_is_closer}, Truth: {item['text_a_is_closer']}")
            time.sleep(0.5)
            
        except Exception as e:
            print(f"Error: {e}")
    
    accuracy = correct / total
    print(f"\nValidation Accuracy: {accuracy:.2%} ({correct}/{total})")
    return accuracy

In [ ]:

val_data = pd.read_json('../Data/SemEval2026-Task_4-dev-v1/dev_track_a.jsonl', lines=True)
val_data = val_data.to_dict(orient='records')
evaluate_validation(val_data)


Validation 1/200...
Extracting symbolic elements...
Prediction: False, Truth: False

Validation 2/200...
Extracting symbolic elements...
Prediction: False, Truth: True

Validation 3/200...
Extracting symbolic elements...
Prediction: False, Truth: False

Validation 4/200...
Extracting symbolic elements...
Prediction: True, Truth: False

Validation 5/200...
Extracting symbolic elements...
Prediction: False, Truth: False

Validation 6/200...
Extracting symbolic elements...
Prediction: False, Truth: False

Validation 7/200...
Extracting symbolic elements...
Prediction: False, Truth: False

Validation 8/200...
Extracting symbolic elements...
Prediction: False, Truth: False

Validation 9/200...
Extracting symbolic elements...
Prediction: True, Truth: True

Validation 10/200...
Extracting symbolic elements...
Prediction: True, Truth: True

Validation 11/200...
Extracting symbolic elements...
Prediction: False, Truth: True

Validation 12/200...
Extracting symbolic elements...
Prediction: Fals